# 0416 · Prueba única con la reserva (septiembre a noviembre de 2026)

**Qué hace.** Evalúa **una sola vez** la decisión registrada en `P4b_decision_v2.json` (C2) sobre guías que ningún cuaderno de modelado había visto.

| Paso | Qué pasa |
| --- | --- |
| 1. Guardas | Existe la decisión; la base de desarrollo es la misma que se validó (SHA-256); la prueba no se ha corrido antes |
| 2. Desarrollo | `V4/MODELADO/base_modelado.parquet` (0411 v2): todo lo conocido antes del 01-09-2026 |
| 3. Prueba | Base de la reserva (`V4/PRUEBA/`, de 0305_GuiasReserva y 0330_BaseReservaV4), con los mismos filtros de 0411 v2 y T0 desde el 01-09 hasta el corte de la reserva menos 86,8 h |
| 4. Historia | Agregados por zona×franja y por cliente congelados el 01-09 (en la prueba solo cuentan desenlaces de desarrollo) |
| 5. Modelos | Los de la decisión (Random Forest y LightGBM L1, con sus parámetros), la logística/lineal, la lineal de mediana y las referencias, entrenados con todo el desarrollo |
| 6. Resultados | Métricas, capacidad fija k = 250/500/1.000, H1 con IC 95 % por días, por mes, por segmento y por día de la semana |
| 7. Para 0420 | Modelos, datos de prueba sin guía y muestra para SHAP en `V4/MODELOS/` |

**Población de prueba.** Las guías de la reserva son las **admitidas** desde el 01-09 (así las extrajo 0110). Las admitidas a fines de agosto con salida en septiembre no entran, porque su desenlace no está en la reserva; se declara en el documento.

**Regla.** No se cambia nada de la decisión después de ver estos resultados. Para ejecutar, poner `EJECUTAR_PRUEBA = True`. Al terminar se escribe `P4d_EJECUTADA.json` y el cuaderno ya no vuelve a correr.

In [0]:
%pip install "duckdb<2" lightgbm xgboost
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 130.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 130.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 172.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 159.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [lightgbm]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, json, glob, sys, time, datetime, hashlib, warnings, gc
import numpy as np, pandas as pd, duckdb, joblib
import lightgbm as lgb, xgboost as xgb, sklearn
from sklearn.linear_model import QuantileRegressor
from sklearn.metrics import mean_absolute_error, average_precision_score
warnings.filterwarnings("ignore")   # E1: sin avisos que impriman rutas del espacio de trabajo

# MODULOS/ debe estar junto al cuaderno
from MODULOS.modelado_temporal import (cargar_festivos, tabla_exclusiones, agregar_historia2, construir_modelo, predecir,
    metricas, metricas_por_segmento, tabla_capacidad, alertas_por_capacidad, diferencia_capacidad_por_bloques,
    diferencia_por_bloques, mediana_por_grupo, puntaje_menor_margen, sha256_archivo, huella)

In [0]:
PATH_DATABRICKS  = "/Volumes/proyectogrado/source/source/"
CARPETA_V4       = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_V4}MODELADO/"
CARPETA_PRUEBA   = f"{CARPETA_V4}PRUEBA/"           # base de la reserva (0330_BaseReservaV4) y salidas P4d_*
CARPETA_MODELOS  = f"{CARPETA_V4}MODELOS/"          # para 0420
RUTA_DECISION    = f"{CARPETA_V4}P4b_decision_v2.json"
RUTA_GOLD_PRUEBA = f"{CARPETA_PRUEBA}GOLD_V4_ENVIO.parquet"
RUTA_MARCA       = f"{CARPETA_PRUEBA}P4d_EJECUTADA.json"
RUTA_CALENDARIO  = f"{PATH_DATABRICKS}PARAMETRICAS/DIM.VW_TIEMPO_CALENDARIO_Y_FESTIVOS.parquet"
os.makedirs(CARPETA_MODELOS, exist_ok=True)

INICIO_PRUEBA     = pd.Timestamp("2026-09-01")
B_BOOTSTRAP       = 1000
MAX_FILAS_CUANTIL = 30_000      # igual que 0412 v2
EJECUTAR_PRUEBA   = True       # poner True UNA sola vez

## 1. Guardas

In [0]:
assert EJECUTAR_PRUEBA, "EJECUTAR_PRUEBA = False: la prueba no se ejecuta"
assert os.path.exists(RUTA_DECISION), "No hay P4b_decision_v2.json: primero registrar la decisión (0416_Comparacion_Decision_2)"
assert not os.path.exists(RUTA_MARCA), "La prueba ya se ejecutó (P4d_EJECUTADA.json): no se repite"
DEC = json.load(open(RUTA_DECISION))
SHA_DECISION = hashlib.sha256(open(RUTA_DECISION, "rb").read()).hexdigest()
HUELLA_BASE = sha256_archivo(f"{CARPETA_MODELADO}base_modelado.parquet")
assert HUELLA_BASE == DEC["huella_base_modelado_sha256"], "La base de desarrollo no es la que se validó para la decisión"
CORTE_PRUEBA = pd.Timestamp(json.load(open(f"{CARPETA_PRUEBA}P3_manifiesto.json"))["parametros"]["CORTE_OBSERVACION"])
P99 = float(DEC["p99_t0_a_entrega_h"])
FIN_PRUEBA = CORTE_PRUEBA - pd.Timedelta(hours=P99)
SEMILLA, USAR_VARIABLES_CLIENTE = DEC["semilla"], DEC["usar_variables_cliente"]
CAT, NUM = DEC["variables_categoricas"], DEC["variables_numericas"]
print(f"Decisión registrada el {DEC['fecha_registro']} · SHA-256 {SHA_DECISION}")
print(f"Clasificación: {DEC['modelo_clf']} · regresión: {DEC['modelo_reg']} · k = {DEC['regla_alertas']['k_por_dia']} alertas/día")
print(f"Prueba: T0 desde {INICIO_PRUEBA} hasta {FIN_PRUEBA} (corte de la reserva {CORTE_PRUEBA} − {P99} h)")

Decisión registrada el 2026-11-08T00:46:23-05:00 · SHA-256 8407242a5b3651d0c1d3bdd5d45d6f96d94af543fa4b85b37f7af0767ae6acfd
Clasificación: RandomForest · regresión: LightGBM · k = 500 alertas/día
Prueba: T0 desde 2026-09-01 00:00:00 hasta 2026-11-01 18:57:09 (corte de la reserva 2026-11-05 09:45:09 − 86.8 h)


## 2. Desarrollo (lo que se validó)

In [0]:
dev = pd.read_parquet(f"{CARPETA_MODELADO}base_modelado.parquet")
CFG = json.load(open(f"{CARPETA_MODELADO}configuracion.json"))
TAREAS = [tuple(t) for t in CFG["TAREAS"]]
dev["_prueba"] = False
print(f"Desarrollo: {len(dev):,} guías · T0 de {dev.t0.min()} a {dev.t0.max()}")

Desarrollo: 1,553,980 guías · T0 de 2026-01-02 07:05:52 a 2026-08-24 16:25:10


## 3. Prueba: base de la reserva con los mismos filtros de 0411 v2

Cada filtro se cuenta en `P4d_exclusiones.csv`.

In [0]:
festivos = cargar_festivos(RUTA_CALENDARIO)
todo = duckdb.sql(f"SELECT * FROM read_parquet('{RUTA_GOLD_PRUEBA}')").df()
for col in ["t0", "FECHA_ENVIO", "fecha_entrega", "t_etiqueta_reg", "t_etiqueta_clf"]:
    todo[col] = pd.to_datetime(todo[col])
todo["entrega_sin_hora"] = todo.fecha_entrega.notna() & (todo.fecha_entrega == todo.fecha_entrega.dt.normalize())
SLA_PRINCIPAL = ["SLA_UNICO"] if CFG["SOLO_SLA_UNICO"] else ["SLA_UNICO", "SLA_POR_REGLA"]
pasos = [("Base de la reserva",                 pd.Series(True, index=todo.index)),
         (f"SLA en {' o '.join(SLA_PRINCIPAL)}", todo.origen_sla.isin(SLA_PRINCIPAL)),
         ("Sin devolución ni siniestro",        ~todo.estado_desenlace.isin(["DEVUELTO", "SINIESTRADO"])),
         ("Entrega no anterior a la admisión (A2)", todo.estado_desenlace != "ENTREGA_ANTES_DE_ADMISION"),
         ("No vencido antes de T0",             ~todo.vencido_antes_de_t0.fillna(False).astype(bool)),
         ("Con etiqueta al corte (no abierta sin vencer)", todo.incumplimiento_sla.notna()),
         ("T0 desde el 01-09-2026",             todo.t0 >= INICIO_PRUEBA),
         (f"T0 al menos {P99} h antes del corte de la reserva (C1)", todo.t0 <= FIN_PRUEBA)]
exclusiones, mask = tabla_exclusiones(todo, pasos)
display(exclusiones)
exclusiones.to_csv(f"{CARPETA_PRUEBA}P4d_exclusiones.csv", index=False)
te = todo[mask].copy(); del todo; gc.collect()
s = te.entrega_sin_hora.astype(bool)                       # A1: siguen en clasificación, sin valor de horas
te.loc[s, ["h_ultima_milla", "t_etiqueta_reg"]] = np.nan
te.loc[te.h_ultima_milla < 0, ["h_ultima_milla", "t_etiqueta_reg"]] = np.nan
# Variables derivadas, igual que 0411 v2 (celda 19)
te["franja"] = pd.cut(te.t0.dt.hour, [-1, 5, 11, 17, 23], labels=["MADRUGADA", "MANANA", "TARDE", "NOCHE"]).astype(str)
te["dia_semana"] = te.t0.dt.dayofweek.astype(str)
te["zona_franja"] = te["categoria_zona"].astype(str) + "_" + te["franja"]
te["dia"] = te.t0.dt.date
te["es_festivo"] = te.t0.dt.normalize().isin(festivos).astype(int)
vol = te.LARGO_ENVIO * te.ANCHO_ENVIO * te.ALTO_ENVIO
te["volumen_declarado_cm3"] = vol.where(vol > 0)
if USAR_VARIABLES_CLIENTE:
    te["ID_CLIENTE"] = te["ID_CLIENTE"].astype(str)
for col in CAT: te[col] = te[col].astype(str)
te["_prueba"] = True
print(f"Prueba: {len(te):,} guías · T0 de {te.t0.min()} a {te.t0.max()} · entregas sin hora: {int(s.sum()):,}")

Festivos cargados exitosamente: 404 (columnas: 'FECHA_FESTIVA', 'FECHA_FESTIVA')


paso,guias,excluidas_en_el_paso,tasa_incumplimiento_excluidas
Base de la reserva,749884,0,null
SLA en SLA_UNICO o SLA_POR_REGLA,742277,7607,0.3881648936170213
Sin devolución ni siniestro,726170,16107,null
Entrega no anterior a la admisión (A2),726159,11,null
No vencido antes de T0,434369,291790,0.9996024538195277
Con etiqueta al corte (no abierta sin vencer),427383,6986,null
T0 desde el 01-09-2026,427383,0,null
T0 al menos 86.8 h antes del corte de la reserva (C1),409137,18246,0.16074756110928423


Prueba: 409,137 guías · T0 de 2026-09-01 05:02:18 a 2026-10-31 11:44:28 · entregas sin hora: 2,291


## 4. Historia congelada el 01-09

Se calcula sobre desarrollo y prueba juntos con `historia_hasta = 01-09`: cada guía de desarrollo recibe su historia sin futuro (igual que en validación) y cada guía de prueba solo ve desenlaces de desarrollo conocidos antes del 01-09.

In [0]:
cols = sorted(set(dev.columns) & set(te.columns))
d = pd.concat([dev[cols], te[cols]], ignore_index=True); del dev; gc.collect()
H = agregar_historia2(d, str(INICIO_PRUEBA), USAR_VARIABLES_CLIENTE)
for k, v in H.items(): d[k] = v
faltan = set(CAT + NUM) - set(d.columns)
assert not faltan, f"Faltan variables en la prueba: {faltan}"
print("Variables completas. Guías en prueba con historia de su zona×franja:",
      f"{d.loc[d._prueba, 'n_hist'].gt(0).mean():.1%}")

Variables completas. Guías en prueba con historia de su zona×franja: 100.0%


## 5. Entrenamiento con todo el desarrollo y predicción en la prueba (una sola vez)

In [0]:
PR, MODELOS_FINALES = [], {}
for tarea, y_col, t_et in TAREAS:
    tr = d.index[~d._prueba & d[y_col].notna() & (d.t0 < INICIO_PRUEBA) & (d[t_et] < INICIO_PRUEBA)]
    ix = d.index[d._prueba & d[y_col].notna()]
    Xtr, ytr, Xte, yte = d.loc[tr, CAT + NUM], d.loc[tr, y_col], d.loc[ix, CAT + NUM], d.loc[ix, y_col]
    base = pd.DataFrame({"tarea": tarea, "y": yte.values, "dia": d.loc[ix, "dia"].astype(str).values,
                         "mes": d.loc[ix, "t0"].dt.strftime("%Y-%m").values, "dia_semana": d.loc[ix, "t0"].dt.dayofweek.values,
                         "categoria_zona": d.loc[ix, "categoria_zona"].values, "franja": d.loc[ix, "franja"].values,
                         "fila": ix.values})
    t_ini = time.time()
    nombre = DEC[f"modelo_{tarea}"]
    m = construir_modelo(tarea, nombre, NUM, CAT, DEC[f"params_{tarea}"], SEMILLA).fit(Xtr, ytr)
    MODELOS_FINALES[tarea] = dict(modelo=m, nombre=nombre, tr=tr, ix=ix, y_col=y_col)
    PR.append(base.assign(modelo=nombre, pred=predecir(m, Xte, tarea)))
    if nombre != "Lineal":
        lin = construir_modelo(tarea, "Lineal", NUM, CAT, {}, SEMILLA).fit(Xtr, ytr)
        PR.append(base.assign(modelo="Lineal", pred=predecir(lin, Xte, tarea)))
    if tarea == "clf":
        PR.append(base.assign(modelo="Referencia", pred=np.full(len(ix), ytr.mean())))
        PR.append(base.assign(modelo="Referencia_tasa_historica", pred=d.loc[ix, "hist_inc_zona_franja"].fillna(ytr.mean()).values))
        PR.append(base.assign(modelo="Referencia_menor_margen", pred=puntaje_menor_margen(d.loc[ix, "margen_sla_en_t0_h"].values)))
    else:
        PR.append(base.assign(modelo="Referencia", pred=d.loc[ix, "hist_h_zona_franja"].fillna(ytr.mean()).values))
        PR.append(base.assign(modelo="Referencia_mediana",
                              pred=mediana_por_grupo(ytr.values, d.loc[tr, "zona_franja"].values, d.loc[ix, "zona_franja"].values)))
        sub = np.arange(len(Xtr))
        if len(sub) > MAX_FILAS_CUANTIL:
            sub = np.sort(np.random.default_rng(SEMILLA).choice(sub, MAX_FILAS_CUANTIL, replace=False))
        mq = construir_modelo("reg", "Lineal", NUM, CAT, {}, SEMILLA)
        mq.set_params(modelo=QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs")).fit(Xtr.iloc[sub], ytr.iloc[sub])
        PR.append(base.assign(modelo="Lineal_mediana", pred=predecir(mq, Xte, "reg")))
    print(f"{tarea}: entrenamiento {len(tr):,} · prueba {len(ix):,} · {nombre} ({time.time() - t_ini:.0f} s)")
PR = pd.concat(PR, ignore_index=True)

clf: entrenamiento 1,553,980 · prueba 409,137 · RandomForest (187 s)
reg: entrenamiento 1,541,316 · prueba 406,259 · LightGBM (195 s)


## 6. Resultados

### 6.1 Métricas generales y por mes

In [0]:
def tabla(pr, por):
    return pd.DataFrame([{**dict(zip(por, k)), **metricas(k[0], g.y, g.pred)} for k, g in pr.groupby(por, sort=False)])
general = tabla(PR, ["tarea", "modelo"]);       display(general.round(4))
por_mes = tabla(PR, ["tarea", "modelo", "mes"]); display(por_mes.round(4))
general.to_csv(f"{CARPETA_PRUEBA}P4d_metricas.csv", index=False)
por_mes.to_csv(f"{CARPETA_PRUEBA}P4d_metricas_por_mes.csv", index=False)

tarea,modelo,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
clf,RandomForest,409137,0.0754,0.2672,0.695,0.0639,0.2453,null,null,null,null,null,null
clf,Lineal,409137,0.0754,0.2432,0.6907,0.0642,0.2443,null,null,null,null,null,null
clf,Referencia,409137,0.0754,0.0754,0.5,0.0697,0.2674,null,null,null,null,null,null
clf,Referencia_tasa_historica,409137,0.0754,0.1494,0.5955,0.0663,0.2539,null,null,null,null,null,null
clf,Referencia_menor_margen,409137,0.0754,0.1177,0.5878,0.132,0.4419,null,null,null,null,null,null
reg,LightGBM,406259,null,null,null,null,null,6.2107,20.6697,7.7892,36.1149,66.0792,0.9744
reg,Lineal,406259,null,null,null,null,null,8.1112,20.6858,16.9329,34.2803,127.0997,0.9744
reg,Referencia,406259,null,null,null,null,null,7.7252,20.6199,14.207,35.2905,122.089,0.9744
reg,Referencia_mediana,406259,null,null,null,null,null,6.3781,21.0467,7.3631,39.7377,64.0481,0.9744
reg,Lineal_mediana,406259,null,null,null,null,null,6.4135,21.1011,7.6844,39.5604,62.62,0.9744


tarea,modelo,mes,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
clf,RandomForest,2026-09,202259,0.0623,0.2787,0.7161,0.0531,0.2139,null,null,null,null,null,null
clf,RandomForest,2026-10,206878,0.0881,0.2605,0.6776,0.0745,0.276,null,null,null,null,null,null
clf,Lineal,2026-09,202259,0.0623,0.2339,0.698,0.0538,0.2122,null,null,null,null,null,null
clf,Lineal,2026-10,206878,0.0881,0.2519,0.6819,0.0744,0.2756,null,null,null,null,null,null
clf,Referencia,2026-09,202259,0.0623,0.0623,0.5,0.0585,0.234,null,null,null,null,null,null
clf,Referencia,2026-10,206878,0.0881,0.0881,0.5,0.0806,0.3,null,null,null,null,null,null
clf,Referencia_tasa_historica,2026-09,202259,0.0623,0.1342,0.5888,0.0556,0.2211,null,null,null,null,null,null
clf,Referencia_tasa_historica,2026-10,206878,0.0881,0.1638,0.5993,0.0767,0.2859,null,null,null,null,null,null
clf,Referencia_menor_margen,2026-09,202259,0.0623,0.1018,0.6019,0.1271,0.4313,null,null,null,null,null,null
clf,Referencia_menor_margen,2026-10,206878,0.0881,0.1312,0.5765,0.1368,0.4522,null,null,null,null,null,null


### 6.2 Capacidad fija (regla de la decisión)

In [0]:
K = DEC["regla_alertas"]["k_por_dia"]; KS = DEC["regla_alertas"]["sensibilidad_k"]
pc = PR[PR.tarea == "clf"]
capacidad = pd.concat([tabla_capacidad(g.y.values, g.pred.values, g.dia.values, KS).assign(modelo=m)
                       for m, g in pc.groupby("modelo", sort=False)], ignore_index=True)
display(capacidad.round(3)); capacidad.to_csv(f"{CARPETA_PRUEBA}P4d_capacidad.csv", index=False)
elegido = DEC["modelo_clf"]
pa = pc[pc.modelo == elegido]
dif_cap = []
for otro in [o for o in ["Lineal", "Referencia_tasa_historica", "Referencia_menor_margen"] if o != elegido]:
    pb = pc[pc.modelo == otro]; assert (pa.fila.values == pb.fila.values).all()
    for k in KS:
        dif_cap.append({"comparacion": f"{elegido} - {otro}", "k_por_dia": k,
                        **diferencia_capacidad_por_bloques(pa.y.values, pa.pred.values, pb.pred.values, pa.dia.values, k, B_BOOTSTRAP, SEMILLA)})
dif_cap = pd.DataFrame(dif_cap); display(dif_cap.round(4)); dif_cap.to_csv(f"{CARPETA_PRUEBA}P4d_capacidad_diferencias.csv", index=False)

k_por_dia,alertas_por_dia_media,exhaustividad,precision,incumplimientos_detectados,incumplimientos_omitidos,falsas_alertas,modelo
250,250.0,0.147,0.336,4542,26288,8958,RandomForest
500,500.0,0.214,0.245,6605,24225,20395,RandomForest
1000,984.944,0.321,0.186,9891,20939,43296,RandomForest
250,250.0,0.133,0.303,4097,26733,9403,Lineal
500,500.0,0.201,0.23,6209,24621,20791,Lineal
1000,984.944,0.306,0.177,9426,21404,43761,Lineal
250,250.0,0.029,0.066,888,29942,12612,Referencia
500,500.0,0.061,0.069,1871,28959,25129,Referencia
1000,984.944,0.122,0.071,3757,27073,49430,Referencia
250,250.0,0.111,0.254,3426,27404,10074,Referencia_tasa_historica


comparacion,k_por_dia,diferencia,ic95_inf,ic95_sup,dias
RandomForest - Lineal,250,0.0144,0.0104,0.0191,54
RandomForest - Lineal,500,0.0128,0.0093,0.0173,54
RandomForest - Lineal,1000,0.0151,0.0082,0.0237,54
RandomForest - Referencia_tasa_historica,250,0.0362,0.0292,0.0445,54
RandomForest - Referencia_tasa_historica,500,0.0312,0.0249,0.0386,54
RandomForest - Referencia_tasa_historica,1000,0.0256,0.0137,0.0388,54
RandomForest - Referencia_menor_margen,250,0.0606,0.0451,0.0778,54
RandomForest - Referencia_menor_margen,500,0.0741,0.0475,0.1041,54
RandomForest - Referencia_menor_margen,1000,0.1092,0.0651,0.151,54


### 6.3 H1 en la prueba: diferencias con IC 95 % por días

AP positiva o MAE negativa favorecen al modelo elegido. En regresión, la pérdida L1 se compara también con las referencias de mediana (C3).

In [0]:
difs = []
pares = [("clf", DEC["modelo_clf"], o) for o in ["Lineal", "Referencia_tasa_historica", "Referencia_menor_margen"]] + \
        [("reg", DEC["modelo_reg"], o) for o in ["Lineal", "Referencia", "Lineal_mediana", "Referencia_mediana"]]
for tarea, a, b in [x for x in pares if x[1] != x[2]]:
    A, Bm = PR[(PR.tarea == tarea) & (PR.modelo == a)], PR[(PR.tarea == tarea) & (PR.modelo == b)]
    assert (A.fila.values == Bm.fila.values).all()
    met = average_precision_score if tarea == "clf" else mean_absolute_error
    difs.append({"tarea": tarea, "metrica": "AP" if tarea == "clf" else "MAE_h", "comparacion": f"H1 · {a} - {b} (prueba)",
                 **diferencia_por_bloques(A.y.values, A.pred.values, Bm.pred.values, A.dia.values, met, B=B_BOOTSTRAP, semilla=SEMILLA)})
difs = pd.DataFrame(difs)
difs["favorece_al_elegido"] = np.where(difs.metrica == "AP", difs.ic95_inf > 0, difs.ic95_sup < 0)
display(difs.round(4)); difs.to_csv(f"{CARPETA_PRUEBA}P4d_diferencias.csv", index=False)

tarea,metrica,comparacion,diferencia,ic95_inf,ic95_sup,dias,favorece_al_elegido
clf,AP,H1 · RandomForest - Lineal (prueba),0.024,0.0124,0.0356,54,true
clf,AP,H1 · RandomForest - Referencia_tasa_historica (prueba),0.1177,0.0879,0.145,54,true
clf,AP,H1 · RandomForest - Referencia_menor_margen (prueba),0.1495,0.1081,0.1933,54,true
reg,MAE_h,H1 · LightGBM - Lineal (prueba),-1.9005,-2.1405,-1.6684,54,true
reg,MAE_h,H1 · LightGBM - Referencia (prueba),-1.5145,-1.7111,-1.2907,54,true
reg,MAE_h,H1 · LightGBM - Lineal_mediana (prueba),-0.2028,-0.2659,-0.1535,54,true
reg,MAE_h,H1 · LightGBM - Referencia_mediana (prueba),-0.1673,-0.2447,-0.1074,54,true


### 6.4 Por segmento y por día de la semana

In [0]:
seg = metricas_por_segmento(PR[PR.modelo.isin([DEC["modelo_clf"], DEC["modelo_reg"], "Lineal"])], ["categoria_zona", "franja"])
seg.to_csv(f"{CARPETA_PRUEBA}P4d_por_segmento.csv", index=False)
sab = pa.assign(alerta_k=alertas_por_capacidad(pa.pred.values, pa.dia.values, K))
tabla_dia = sab.groupby("dia_semana").agg(guias=("y", "size"), dias=("dia", "nunique"), prevalencia=("y", "mean"),
                                          p_media=("pred", "mean"), alertas_k=("alerta_k", "sum"), incumplimientos=("y", "sum"),
                                          detectados=("alerta_k", lambda a: int((a & (sab.loc[a.index, "y"] == 1)).sum())))
tabla_dia["exhaustividad_k"] = tabla_dia.detectados / tabla_dia.incumplimientos.replace(0, np.nan)
tabla_dia.index = tabla_dia.index.map(dict(enumerate(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])))
display(tabla_dia.round(3)); tabla_dia.to_csv(f"{CARPETA_PRUEBA}P4d_dia_semana.csv")

guias,dias,prevalencia,p_media,alertas_k,incumplimientos,detectados,exhaustividad_k
37583,8,0.047,0.081,4000,1781.0,419,0.235
80169,9,0.054,0.094,4500,4354.0,766,0.176
84495,9,0.078,0.095,4500,6552.0,1034,0.158
75309,9,0.067,0.096,4500,5055.0,816,0.161
74859,9,0.067,0.097,4500,5049.0,805,0.159
56204,9,0.143,0.172,4500,8034.0,2760,0.344
518,1,0.01,0.046,500,5.0,5,1.0


## 7. Guardar para 0420 (interpretabilidad y evaluación espacial)

Modelo final, datos de prueba sin guía (con `zona_urbana` para la evaluación espacial) y una muestra de entrenamiento para SHAP.

In [0]:
for tarea, t in MODELOS_FINALES.items():
    joblib.dump(t["modelo"], f"{CARPETA_MODELOS}modelo_final_{tarea}.joblib")
    X = d.loc[t["ix"], CAT + NUM].copy()
    p = PR[(PR.tarea == tarea) & (PR.modelo == t["nombre"])]
    X["y"], X["pred"] = p.y.values, p.pred.values
    X["dia"] = d.loc[t["ix"], "dia"].astype(str).values
    if "zona_urbana" in d: X["zona_urbana"] = d.loc[t["ix"], "zona_urbana"].values
    X.to_parquet(f"{CARPETA_MODELOS}prueba_{tarea}.parquet", index=False)
    s = d.loc[t["tr"], CAT + NUM].sample(min(50_000, len(t["tr"])), random_state=SEMILLA)
    s[t["y_col"]] = d.loc[s.index, t["y_col"]].values
    s.to_parquet(f"{CARPETA_MODELOS}muestra_entrenamiento_{tarea}.parquet", index=False)
print("Guardado en", CARPETA_MODELOS)

Guardado en /Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_11/V4/MODELOS/


## 8. Manifiesto y marca de ejecución

In [0]:
manifiesto = {"fecha_ejecucion": datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=-5))).isoformat(timespec="seconds"),
              "decision_fecha_registro": DEC["fecha_registro"], "decision_sha256": SHA_DECISION,
              "huella_base_modelado_sha256": HUELLA_BASE, "gold_reserva": huella(RUTA_GOLD_PRUEBA),
              "corte_reserva": str(CORTE_PRUEBA), "inicio_prueba": str(INICIO_PRUEBA), "fin_prueba": str(FIN_PRUEBA),
              "guias_prueba": {t: int(len(v["ix"])) for t, v in MODELOS_FINALES.items()},
              "guias_entrenamiento": {t: int(len(v["tr"])) for t, v in MODELOS_FINALES.items()},
              "versiones": {"python": sys.version.split()[0], "sklearn": sklearn.__version__, "lightgbm": lgb.__version__,
                            "pandas": pd.__version__, "duckdb": duckdb.__version__},
              "exclusiones": exclusiones.to_dict("records"), "B_bootstrap": B_BOOTSTRAP,
              "salidas": sorted(os.path.basename(f) for f in glob.glob(f"{CARPETA_PRUEBA}P4d_*"))}
with open(f"{CARPETA_PRUEBA}P4d_manifiesto.json", "w") as f: json.dump(manifiesto, f, indent=2, default=str)
with open(RUTA_MARCA, "w") as f: json.dump({"ejecutada": manifiesto["fecha_ejecucion"], "decision_sha256": SHA_DECISION}, f, indent=2)
print("Prueba ejecutada y marcada:", manifiesto["fecha_ejecucion"])

Prueba ejecutada y marcada: 2026-11-08T09:06:57-05:00


## Qué entregar

De `V4/PRUEBA/`: `P4d_exclusiones.csv`, `P4d_metricas.csv`, `P4d_metricas_por_mes.csv`, `P4d_capacidad.csv`, `P4d_capacidad_diferencias.csv`, `P4d_diferencias.csv`, `P4d_por_segmento.csv`, `P4d_dia_semana.csv`, `P4d_manifiesto.json`, los `P3_*` de la reserva y `P4d_EJECUTADA.json`.